In [ ]:
import pandas as pd
import random
import os
from bigcrittercolor.helpers import _getBCCIDs, _bprint

def _getSpeciesBalancedIDs2(type="image", n_per_species=200, usa_only=False, 
                             print_steps=True, print_details=True, data_folder=''):
    
    # USA bounding box: ((min_lat, min_lon), (max_lat, max_lon))
    lat_lon_box = ((24.396308, -124.848974), (49.384358, -66.885444))
    
    # Damselfly genera to exclude (Zygoptera)
    damselfly_genera = [
        "Argia", "Enallagma", "Ischnura", "Nehalennia", "Chromagrion", 
        "Amphiagrion", "Lestes", "Sympecma", "Archilestes", "Calopteryx", 
        "Hetaerina", "Mnesarete", "Psolodesmus", "Telebasis"
    ]
    
    _bprint(print_steps, "Started _getSpeciesBalancedIDs...")
    
    # Load records and filter to downloaded images
    csv_path = os.path.join(data_folder, "records.csv")
    data = pd.read_csv(csv_path)
    
    _bprint(print_steps, "Getting all downloaded image ids...")
    downloaded_ids = _getBCCIDs(type=type, data_folder=data_folder)
    filtered_data = data[data['img_id'].isin(downloaded_ids)]
    
    # Filter by USA bounding box if requested
    if usa_only:
        (min_lat, min_lon), (max_lat, max_lon) = lat_lon_box
        filtered_data = filtered_data[
            (filtered_data['latitude'] >= min_lat) & 
            (filtered_data['latitude'] <= max_lat) &
            (filtered_data['longitude'] >= min_lon) & 
            (filtered_data['longitude'] <= max_lon)
        ]
        _bprint(print_steps, f"Filtered to {len(filtered_data)} USA-only records.")
    
    # Exclude damselfly genera
    filtered_data = filtered_data[~filtered_data['genus'].isin(damselfly_genera)]
    _bprint(print_steps, f"Excluded damselfly genera. Remaining records: {len(filtered_data)}")
    
    # Keep only species with at least 500 observations
    species_counts = filtered_data['species'].value_counts()
    valid_species = species_counts[species_counts >= 500].index
    filtered_data = filtered_data[filtered_data['species'].isin(valid_species)]
    _bprint(print_steps, f"Filtered to species with ≥500 records. Remaining: {len(filtered_data)}")
    
    # Sample n_per_species from each species
    _bprint(print_steps, "Sampling IDs for each species...")
    balanced_ids = []
    species_count = {}
    
    for species, group in filtered_data.groupby('species'):
        sampled_ids = group['img_id'].sample(
            min(n_per_species, len(group)), 
            random_state=1
        ).tolist()
        balanced_ids.extend(sampled_ids)
        species_count[species] = len(sampled_ids)
    
    if print_details:
        for species, count in species_count.items():
            print(f"{species}: {count} ids")
    
    _bprint(print_steps, "Finished _getSpeciesBalancedIDs.")
    return balanced_ids


def saveSpeciesBalancedImageIDs2(n_per_species=200, print_steps=True, 
                                  print_details=True, data_folder=''):
    balanced_ids = _getSpeciesBalancedIDs2(
        type="image",
        n_per_species=n_per_species,
        usa_only=True,
        print_steps=print_steps,
        print_details=print_details,
        data_folder=data_folder
    )
    
    _bprint(print_steps, "Writing species-balanced ids...")
    balanced_ids_path = os.path.join(data_folder, "species_balanced_ids.txt")
    with open(balanced_ids_path, 'w') as f:
        for img_id in balanced_ids:
            f.write(f"{img_id}\n")
            
saveSpeciesBalancedImageIDs2(n_per_species=2000, print_steps=True, print_details=True,data_folder="/blue/guralnick/jacob.idec/bcc_odonates")

In [ ]:
import os

from bigcrittercolor.helpers import _getBCCIDs, _bprint

# stage can be "image", "mask", "segment"
def readSpeciesBalancedIDs(stage_type="image",print_steps=True,data_folder=''):
    _bprint(print_steps, "Reading previously saved species-balanced ids...")
    # Read the balanced ids from the text file
    balanced_ids_path = os.path.join(data_folder, "species_balanced_ids.txt")
    with open(balanced_ids_path, 'r') as f:
        balanced_ids = [line.strip() for line in f.readlines()]

    if stage_type == "image":
        return balanced_ids

    _bprint(print_steps, f"Filtering for species-balanced ids that have reached the {stage_type} stage...")
    # stage is either "mask" or "segment"
    stage_ids = _getBCCIDs(type=stage_type,data_folder=data_folder)

    # get balanced ids that have actually reached the stage
    balanced_ids = set(balanced_ids).intersection(set(stage_ids))

    balanced_ids = list(balanced_ids)

    return balanced_ids

ids = readSpeciesBalancedIDs(stage_type="image",data_folder="/blue/guralnick/jacob.idec/bcc_odonates")

from bigcrittercolor import inferMasks

inferMasks(img_ids=ids,data_folder="/blue/guralnick/jacob.idec/bcc_odonates",gd_gpu=True,sam_gpu=True,
           aux_segmodel_location="/blue/guralnick/jacob.idec/aux_segmenter_unetpp_dragonflies.pt",print_details=False,show_indv=False)

In [ ]:
from bigcrittercolor import filterExtractSegments
from bigcrittercolor.project import readSpeciesBalancedIDs

ids = readSpeciesBalancedIDs(stage_type="mask", data_folder="/blue/guralnick/jacob.idec/bcc_odonates")
filterExtractSegments(img_ids=ids,used_aux_segmodel=True,batch_size=30000,
                      cluster_params_dict={'algo':"gaussian_mixture",'pca_n':5,'n':10,'scale':"standard",
                                          'seed':0},
                      data_folder="/blue/guralnick/jacob.idec/bcc_odonates")

In [ ]:
from bigcrittercolor import writeBasicColorMetrics
from bigcrittercolor.project import readSpeciesBalancedIDs


ids = readSpeciesBalancedIDs(stage_type="segment",data_folder="/blue/guralnick/jacob.idec/bcc_odonates")
writeBasicColorMetrics(
    img_ids=ids, 
    get_color_metrics=False, 
    get_shape_texture_metrics=False,
    threshold_metrics=[
        ("hls", 1, 0.3, "below"),
        ("hls", 1, 0.2, "below")
    ], 
    show=False,
    data_folder="/blue/guralnick/jacob.idec/bcc_odonates"
)